# Corneal Transplant Surgery Analysis

Clean, consolidated analysis notebook.  
All results are saved to **`Nisha_Analysis_Results.xlsx`** at the end (one sheet per analysis table).

---
## 0 · Setup & Configuration

In [25]:
import pandas as pd
import numpy as np
import regex as re
from datetime import datetime
from tqdm.notebook import tqdm
import openpyxl

pd.set_option('display.max_columns', None)

# ── Paths ──
INPUT_FILE  = r"C:\Users\MohammadFarhan\Desktop\Raksha\morbidity\Given Corn Tran Surg Px  Adv and FUP Data Details-April2026.xlsb"
OUTPUT_FILE = r"C:\Users\MohammadFarhan\Desktop\Raksha\morbidity\Morbidity_Analysis_Results_Apr26.xlsx"

# ── Campus sheets ──
CAMPUS_SHEETS = [
    'KAR Campus Data',
    'KVC Campus Data',
    'GMR Campus Data',
    'MTC Campus Data',
    'YSR Campus Data',
]

# ── Follow-up windows (days) ──
FOLLOW_UP_PERIODS = {
    "1D": (0, 3),
    "1W": (4, 10),
    "1M": (11, 45),
    "3M": (46, 135)
}

# ── Visual acuity special-value mapping (Snellen → LogMAR) ──
VA_SPECIAL_MAP = {
    "HM": 2.3, "HM+": 2.3,
    "CF CF": 1.9, "FFL": 1.9,
    "CF": 1.6, "CF 1M": 1.6, "CSM": 1.6,
    "CF 2M": 1.5,
    "NPL": 3,
    "PL": 2.7, "PL+": 2.7, "PL+ PR ACC": 2.7,
    "PL+ PR ACCURATE": 2.7, "PL+ PR INA": 2.7,
    "PL+ PR INACCURATE": 2.7,
}

# ── Repeat surgery procedure sets ──
REPEAT_KP_PROCS = {
    "THERAPUETIC PENETRATING KERATOPLASTY (TH PK)",
    "INTRAOCULAR ANTIBIOTIC INJECTION (IOAB),THERAPUETIC PENETRATING KERATOPLASTY (TH PK)",
    "DESCEMETS STRIPPING AUTOMATED ENDOTHELIAL KERATOPLASTY (DSAEK)",
    "ANTERIOR VITRECTOMY,THERAPUETIC PENETRATING KERATOPLASTY (TH PK)",
    "PENETRATING KERATOPLASTY (PK)",
    "DESCEMET MEMBRANE ENDOTHLIAL KERATOPLASTY (DMEK)",
    "ECCE + IOL,PENETRATING KERATOPLASTY (PK),TARSORRAPHY",
}

# ── MRNO prefix corrections ──
PREFIX_MAP = {"PN": "P", "NP": "N", "NPC": "CC", "PNC": "CC", "NPN": "N", "PNP": "P"}

# ── Age bins ──
AGE_BINS   = [0, 18, 40, 60, 80, np.inf]
AGE_LABELS = ["<18yr", "18-40yr", "41-60yr", "61-80yr", "80+yr"]

# ── Collector for all Excel sheets ──
EXCEL_SHEETS: dict[str, pd.DataFrame] = {}

print("✓ Configuration loaded.")

✓ Configuration loaded.


---
## 1 · Helper Functions

In [26]:
# ── Date fixing (handles mixed text + Excel serial dates) ──
def fix_date(df: pd.DataFrame, column: str) -> pd.DataFrame:
    df = df.copy()
    parsed = pd.to_datetime(df[column], errors='coerce')
    numeric = pd.to_numeric(df[column], errors='coerce')
    valid_numeric = numeric.where(numeric.between(1, 100000))
    excel_dates = pd.Timestamp("1899-12-30") + pd.to_timedelta(valid_numeric, unit="D", errors='coerce')
    df[column] = parsed.fillna(excel_dates)
    return df


# ── Standardise object columns (strip + upper) ──
def clean_object_columns(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = df[col].apply(lambda x: x.strip().upper() if isinstance(x, str) else x)
    return df


# ── Correct MRNO prefixes ──
def correct_mrno(mrno: str) -> str:
    match = re.search(r"(?:.*-)?([A-Z]+)(\d+)$", mrno)
    if match:
        prefix, digits = match.groups()
        new_prefix = PREFIX_MAP.get(prefix, prefix)
        return re.sub(r"([A-Z]+)(\d+)$", new_prefix + digits, mrno)
    return mrno


# ── Build VA → LogMAR lookup from data ──
def build_va_logmar_dict(df: pd.DataFrame, va_cols: list[str]) -> dict:
    unique_vals = set()
    for col in va_cols:
        unique_vals.update(df[col].dropna().unique())

    mapping = {}
    for va in unique_vals:
        va_str = str(va).strip().upper()
        if va_str in VA_SPECIAL_MAP:
            mapping[va_str] = VA_SPECIAL_MAP[va_str]
        elif "/" in va_str:
            try:
                cleaned = va_str.replace("P", "")
                num, den = cleaned.split("/")
                mapping[va_str] = round(-1 * np.log10(int(num) / int(den)), 4)
            except (ValueError, ZeroDivisionError):
                mapping[va_str] = np.nan
        else:
            mapping[va_str] = np.nan
    return mapping


# ── Generic count summary ──
def count_by(df: pd.DataFrame, col: str) -> pd.DataFrame:
    return (
        df.groupby(col).size()
          .reset_index(name='Count')
          .sort_values('Count', ascending=False)
    )


# ── Graft summary (overall or grouped) ──
def make_graft_summary(df: pd.DataFrame, group_cols: list[str] | None = None) -> pd.DataFrame:
    if group_cols is None:
        summary = df.groupby('graft_health_text').size().reset_index(name='Count')
        total = len(df)
        summary['Total Performed'] = total
        summary['Percentage'] = (summary['Count'] / total * 100).round(2)
        return summary

    summary = df.groupby(group_cols + ['graft_health_text']).size().reset_index(name='Count')
    totals = df.groupby(group_cols).size().reset_index(name='Total Performed')
    summary = summary.merge(totals, on=group_cols, how='left')
    summary['Percentage'] = (summary['Count'] / summary['Total Performed'] * 100).round(2)
    return summary


# ── Combine graft period summaries across follow-up windows ──
def combine_graft_periods(summary_dict: dict, group_cols: list[str],
                          surgery_df: pd.DataFrame) -> pd.DataFrame:
    combined = None
    for period, sdf in summary_dict.items():
        temp = sdf[group_cols + ['graft_health_text', 'Count', 'Percentage']].copy()
        temp.rename(columns={'Count': f'{period} Count', 'Percentage': f'{period} %'}, inplace=True)
        if combined is None:
            combined = temp
        else:
            combined = combined.merge(temp, on=group_cols + ['graft_health_text'], how='outer')

    # Total performed
    if len(group_cols) == 0:
        combined['Total Performed'] = len(surgery_df)
    else:
        totals = surgery_df.groupby(group_cols).size().reset_index(name='Total Performed')
        combined = combined.merge(totals, on=group_cols, how='left')

    # Clean up
    for c in [c for c in combined.columns if c.endswith('Count')]:
        combined[c] = combined[c].fillna(0).astype(int)
    for c in [c for c in combined.columns if c.endswith('%')]:
        combined[c] = combined[c].fillna(0).round(2)

    ordered = group_cols + ['graft_health_text', 'Total Performed']
    for period in FOLLOW_UP_PERIODS:
        ordered.extend([f'{period} Count', f'{period} %'])
    return combined[ordered]


# ── Adherence summary (overall or grouped) ──
def adherence_summary(adherence_df: pd.DataFrame, group_col: str | None = None) -> dict:
    summaries = {}
    for period in FOLLOW_UP_PERIODS:
        if group_col is None:
            temp = (
                adherence_df[period]
                .value_counts()
                .reindex(["YES", "NO", "PENDING"], fill_value=0)
                .rename_axis("Status")
                .reset_index(name="Count")
            )
            temp["Percentage"] = (temp["Count"] / temp["Count"].sum() * 100).round(1)
        else:
            temp = (
                adherence_df
                .groupby([group_col, period]).size()
                .unstack(fill_value=0)
                .reindex(columns=["YES", "NO", "PENDING"], fill_value=0)
            )
            temp["Total"] = temp.sum(axis=1)
            for s in ["YES", "NO", "PENDING"]:
                temp[f"{s}_%"] = (temp[s] / temp["Total"] * 100).round(1)
            temp = temp.reset_index()
        summaries[period] = temp
    return summaries


# ── Combine adherence period summaries ──
def combine_adherence_periods(summary_dict: dict, adherence_df: pd.DataFrame,
                              group_cols: list[str] | None = None) -> pd.DataFrame:
    if group_cols is None:
        group_cols = []

    combined = None
    for period, sdf in summary_dict.items():
        if len(group_cols) == 0:
            temp = sdf[['Status', 'Count', 'Percentage']].copy()
            temp.rename(columns={'Count': f'{period} Count', 'Percentage': f'{period} %'}, inplace=True)
            merge_cols = ['Status']
        else:
            # Exclude the per-period 'Total' column to avoid creating duplicate
            # non-key columns when merging across periods. Totals are attached once later.
            temp = sdf[group_cols + ['YES', 'NO', 'PENDING', 'YES_%', 'NO_%', 'PENDING_%']].copy()
            count_long = temp.melt(
                id_vars=group_cols,
                value_vars=['YES', 'NO', 'PENDING'],
                var_name='Status', value_name=f'{period} Count'
            )
            pct_long = temp.melt(
                id_vars=group_cols,
                value_vars=['YES_%', 'NO_%', 'PENDING_%'],
                var_name='Status', value_name=f'{period} %'
            )
            pct_long['Status'] = pct_long['Status'].str.replace('_%', '', regex=False)
            temp = count_long.merge(pct_long, on=group_cols + ['Status'])
            merge_cols = group_cols + ['Status']

        combined = temp if combined is None else combined.merge(temp, on=merge_cols, how='outer')

    # Total surgeries
    if len(group_cols) == 0:
        combined['Total Surgeries'] = len(adherence_df)
    else:
        totals = adherence_df.groupby(group_cols).size().reset_index(name='Total Surgeries')
        combined = combined.merge(totals, on=group_cols, how='left')

    for c in [c for c in combined.columns if c.endswith('Count')]:
        combined[c] = combined[c].fillna(0).astype(int)
    for c in [c for c in combined.columns if c.endswith('%')]:
        combined[c] = combined[c].fillna(0).round(2)

    ordered = group_cols + ['Status', 'Total Surgeries']
    for period in FOLLOW_UP_PERIODS:
        ordered.extend([f'{period} Count', f'{period} %'])
    return combined[ordered]


# ── Categorise repeat surgery ──
def categorize_repeat_surgery(x):
    if pd.isna(x):
        return "Others"
    x = x.strip()
    if x == "REBUBBLING":
        return "REBUBBLING"
    if x == "WOUND RESUTURING":
        return "WOUND_RESUTURING"
    if x in REPEAT_KP_PROCS:
        return "KP"
    return "Others"


# ── VA line-change category ──
def line_category(diff):
    if pd.isna(diff):
        return np.nan
    lines = int(abs(diff) // 0.1)
    if lines == 0:
        return "No Change (<1 Line)"
    elif lines <= 5:
        return f"{lines} Line{'s' if lines > 1 else ''}"
    else:
        return ">5 Lines"


# ── Average visit stats ──
def average_visits(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    return (
        df.groupby(group_cols + ["id"]).size()
          .reset_index(name="No_of_Visits")
          .groupby(group_cols)
          .agg(
              Total_Surgeries=("No_of_Visits", "count"),
              Total_Visits   =("No_of_Visits", "sum"),
              Avg_Visits     =("No_of_Visits", "mean"),
              Median_Visits  =("No_of_Visits", "median"),
              Min_Visits     =("No_of_Visits", "min"),
              Max_Visits     =("No_of_Visits", "max"),
              SD_Visits      =("No_of_Visits", "std"),
          )
          .round({"Avg_Visits": 2, "SD_Visits": 2})
          .reset_index()
    )




# ── Beautify Excel Workbook (openpyxl executive formatting) ──
def beautify_workbook(wb: openpyxl.Workbook) -> openpyxl.Workbook:
    HEADER_FILL = openpyxl.styles.PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
    HEADER_FONT = openpyxl.styles.Font(name="Segoe UI", size=11, bold=True, color="FFFFFF")
    
    DATA_FONT = openpyxl.styles.Font(name="Segoe UI", size=10, bold=False, color="000000")
    BOLD_FONT = openpyxl.styles.Font(name="Segoe UI", size=10, bold=True, color="000000")
    TOTAL_ROW_FILL = openpyxl.styles.PatternFill(start_color="EBF1F5", end_color="EBF1F5", fill_type="solid")
    ALT_ROW_FILL = openpyxl.styles.PatternFill(start_color="F8F9FA", end_color="F8F9FA", fill_type="solid")
    WHITE_ROW_FILL = openpyxl.styles.PatternFill(start_color="FFFFFF", end_color="FFFFFF", fill_type="solid")
    
    THIN_BORDER_SIDE = openpyxl.styles.Side(border_style="thin", color="D9D9D9")
    MEDIUM_BOTTOM_SIDE = openpyxl.styles.Side(border_style="medium", color="1F4E78")
    DOUBLE_BOTTOM_SIDE = openpyxl.styles.Side(border_style="double", color="1F4E78")
    
    DATA_BORDER = openpyxl.styles.Border(left=THIN_BORDER_SIDE, right=THIN_BORDER_SIDE, top=THIN_BORDER_SIDE, bottom=THIN_BORDER_SIDE)
    HEADER_BORDER = openpyxl.styles.Border(left=THIN_BORDER_SIDE, right=THIN_BORDER_SIDE, top=THIN_BORDER_SIDE, bottom=MEDIUM_BOTTOM_SIDE)
    TOTAL_BORDER = openpyxl.styles.Border(left=THIN_BORDER_SIDE, right=THIN_BORDER_SIDE, top=THIN_BORDER_SIDE, bottom=DOUBLE_BOTTOM_SIDE)
    
    TAB_COLORS = {
        "1": "2B579A",  # Surgery Counts (Navy Blue)
        "2": "27AE60",  # Graft Health (Emerald Green)
        "3": "D35400",  # Adherence (Warm Amber)
        "4": "C0392B",  # Failed & Infection (Crimson Red)
        "5": "8E44AD",  # Repeat Surgeries (Purple)
        "6": "2980B9",  # VA Change (Ocean Blue)
    }

    for sheetname in wb.sheetnames:
        ws = wb[sheetname]
        prefix = sheetname.split("_")[0]
        if prefix in TAB_COLORS:
            ws.sheet_properties.tabColor = TAB_COLORS[prefix]
            
        ws.views.sheetView[0].showGridLines = True
        ws.freeze_panes = 'A2'
        
        max_row = ws.max_row
        max_col = ws.max_column
        if max_row == 0 or max_col == 0:
            continue
            
        ws.auto_filter.ref = ws.dimensions
        ws.row_dimensions[1].height = 28
        
        headers = []
        for col in range(1, max_col + 1):
            cell = ws.cell(row=1, column=col)
            header_val = str(cell.value) if cell.value is not None else ""
            headers.append(header_val)
            cell.fill = HEADER_FILL
            cell.font = HEADER_FONT
            cell.alignment = openpyxl.styles.Alignment(horizontal="center", vertical="center", wrap_text=True)
            cell.border = HEADER_BORDER

        for row in range(2, max_row + 1):
            ws.row_dimensions[row].height = 20
            first_cell_val = str(ws.cell(row=row, column=1).value or "").strip().lower()
            is_total_row = first_cell_val in ["overall", "total", "total primary surgeries"] or "total" in first_cell_val
            row_fill = TOTAL_ROW_FILL if is_total_row else (ALT_ROW_FILL if row % 2 == 0 else WHITE_ROW_FILL)
            row_font = BOLD_FONT if is_total_row else DATA_FONT
            row_border = TOTAL_BORDER if is_total_row else DATA_BORDER
            
            for col in range(1, max_col + 1):
                cell = ws.cell(row=row, column=col)
                cell.font = row_font
                cell.fill = row_fill
                cell.border = row_border
                header_name = headers[col - 1] if col - 1 < len(headers) else ""
                val = cell.value
                header_lower = header_name.lower()
                
                if "date" in header_lower or header_lower == "dob":
                    cell.alignment = openpyxl.styles.Alignment(horizontal="center", vertical="center")
                    if val is not None:
                        cell.number_format = "yyyy-mm-dd"
                elif "%" in header_name or "pct" in header_lower or "percentage" in header_lower:
                    cell.alignment = openpyxl.styles.Alignment(horizontal="right", vertical="center")
                    if isinstance(val, (int, float)):
                        if abs(val) > 1.0 or val == 0:
                            cell.number_format = '0.0"%"'
                        else:
                            cell.number_format = '0.0%'
                elif any(k in header_name for k in ["Count", "Total", "Surgeries", "Visits", "Patients", "Lines"]) and not any(k in header_lower for k in ["avg", "sd", "median"]):
                    cell.alignment = openpyxl.styles.Alignment(horizontal="right", vertical="center")
                    if isinstance(val, (int, float)):
                        cell.number_format = '#,##0'
                elif any(k in header_lower for k in ["logmar", "diff"]):
                    cell.alignment = openpyxl.styles.Alignment(horizontal="right", vertical="center")
                    if isinstance(val, (int, float)):
                        cell.number_format = '0.000'
                elif any(k in header_lower for k in ["avg", "sd", "median"]):
                    cell.alignment = openpyxl.styles.Alignment(horizontal="right", vertical="center")
                    if isinstance(val, (int, float)):
                        cell.number_format = '0.00'
                elif header_lower in ["sap_code", "gender", "status", "surg_eye", "age_category"]:
                    cell.alignment = openpyxl.styles.Alignment(horizontal="center", vertical="center")
                else:
                    if isinstance(val, (int, float)):
                        cell.alignment = openpyxl.styles.Alignment(horizontal="right", vertical="center")
                    else:
                        cell.alignment = openpyxl.styles.Alignment(horizontal="left", vertical="center")

        for col in range(1, max_col + 1):
            col_letter = openpyxl.utils.get_column_letter(col)
            max_len = 0
            for row in range(1, max_row + 1):
                val_str = str(ws.cell(row=row, column=col).value or "")
                if len(val_str) > max_len:
                    max_len = len(val_str)
            header_len = len(headers[col - 1]) if col - 1 < len(headers) else 0
            width = max(max_len + 4, header_len + 4, 12)
            ws.column_dimensions[col_letter].width = min(width, 48)
    return wb

print("✓ Helper functions defined.")

✓ Helper functions defined.


---
## 2 · Data Loading & Cleaning

In [27]:
# ── Load all campus sheets and concatenate ──
campus_frames = [
    pd.read_excel(INPUT_FILE, sheet_name=sheet, header=4)
    for sheet in CAMPUS_SHEETS
]
df = pd.concat(campus_frames, ignore_index=True)
del campus_frames

print(f"Loaded {len(df):,} rows from {len(CAMPUS_SHEETS)} campus sheets.")

Loaded 1,924 rows from 5 campus sheets.


In [28]:
# ── Fix date columns ──
DATE_COLS = ["visit_date", "surg_date", "dob", "fup_surg_date"]
for col in DATE_COLS:
    before = df[col].isna().sum()
    df = fix_date(df, col)
    after = df[col].isna().sum()
    print(f"{col}: nulls {before} → {after}")

# ── Standardise text columns ──
df = clean_object_columns(df)

# ── Correct MRNOs ──
df["tp_mrno_corrected"] = df["tp_mrno"].apply(correct_mrno)

# ── Build composite surgery ID ──
df['id'] = (
    df['tp_mrno_corrected'] + '_' +
    df['surg_eye'] + '_' +
    df['surg_proc_group'] + '_' +
    df['surg_date'].dt.strftime('%Y-%m-%d')
)

# ── Drop empty rows & sort ──
df = df.dropna(how='all').dropna(subset=['id']).sort_values(['id', 'surg_date', 'visit_date'])

# ── Compute LogMAR for VA columns ──
VA_COLS = ['ucva', 'pin_hole', 'bcva']
va_map = build_va_logmar_dict(df, VA_COLS)
for col in VA_COLS:
    df[f"{col}_logmar"] = df[col].map(va_map)
df["best_va_logmar"] = df[["ucva_logmar", "pin_hole_logmar", "bcva_logmar"]].min(axis=1)

# ── Derived columns ──
df['age'] = df["visit_date"].dt.year - df["dob"].dt.year
df['surg_age'] = df["surg_date"].dt.year - df["dob"].dt.year
df['days_after_surgery'] = (df['visit_date'] - df['surg_date']).dt.days

print(f"\n✓ Cleaning complete. {df['id'].nunique():,} unique surgery IDs, {df['tp_mrno_corrected'].nunique():,} unique patients.")

visit_date: nulls 0 → 0
surg_date: nulls 0 → 0
dob: nulls 0 → 0
fup_surg_date: nulls 1450 → 1771

✓ Cleaning complete. 321 unique surgery IDs, 319 unique patients.


C:\Users\MohammadFarhan\AppData\Local\Temp\ipykernel_33484\3598553569.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(df[column], errors='coerce')


---
## 3 · Build Core DataFrames

In [29]:
# ── Visit-level dataset ──
VISIT_COLS = [
    'id', 'sap_code', 'tp_mrno_corrected', 'tp_mrno', 'dob', 'age', 'gender',
    'sub_category', 'surg_date', 'surg_eye', 'visit_date', 'days_after_surgery',
    'surg_age', 'best_va_logmar', 'surgeon_name', 'advise_surg',
    'surg_proc_group', 'graft_health_text', 'fup_surg_done', 'fup_surg_date',
    'fup_surg_doctor', 'fup_done_surg_proc',
]
visit_df = df[VISIT_COLS].copy()

# ── Identify first repeat keratoplasty date ──
repeat_mask = (
    (visit_df['visit_date'] > visit_df['surg_date']) &
    (visit_df['advise_surg'] == 'YES') &
    (visit_df['fup_surg_done'] == 'YES') &
    (visit_df['fup_done_surg_proc'].isin(REPEAT_KP_PROCS))
)
repeat_dates = (
    visit_df.loc[repeat_mask]
    .groupby('id', as_index=False)['fup_surg_date'].min()
    .rename(columns={'fup_surg_date': 'first_repeat_keratoplasty_date'})
)
visit_df = visit_df.merge(repeat_dates, on='id', how='left')

# ── Primary-surgery visits (before any repeat KP) ──
visit_df_primary = visit_df[
    visit_df['first_repeat_keratoplasty_date'].isna() |
    (visit_df['visit_date'] < visit_df['first_repeat_keratoplasty_date'])
].copy()

# ── One row per primary surgery ──
surgery_df = visit_df_primary.groupby('id', as_index=False).first()

# ── Age category ──
visit_df_primary["age_category"] = pd.cut(
    visit_df_primary["surg_age"], bins=AGE_BINS, labels=AGE_LABELS, right=True
)

print(f"✓ {len(surgery_df):,} primary surgeries  |  {len(visit_df_primary):,} primary visit rows.")

✓ 321 primary surgeries  |  1,901 primary visit rows.


---
## Analysis 1 · Surgery Counts

In [30]:
total_surgeries = pd.DataFrame({"Metric": ["Total Primary Surgeries"], "Count": [len(surgery_df)]})
gender_summary   = count_by(surgery_df, 'gender')
campus_summary   = count_by(surgery_df, 'sap_code')
surg_type_summary = count_by(surgery_df, 'surg_proc_group')
category_summary = count_by(surgery_df, 'sub_category')
faculty_summary  = count_by(surgery_df, 'surgeon_name')

EXCEL_SHEETS["1_Total"]        = total_surgeries
EXCEL_SHEETS["1_By Gender"]    = gender_summary
EXCEL_SHEETS["1_By Campus"]    = campus_summary
EXCEL_SHEETS["1_By SurgType"]  = surg_type_summary
EXCEL_SHEETS["1_By Category"]  = category_summary
EXCEL_SHEETS["1_By Surgeon"]   = faculty_summary

print("✓ Analysis 1 (Surgery Counts) — 6 sheets queued.")
display(total_surgeries)

✓ Analysis 1 (Surgery Counts) — 6 sheets queued.


,Metric,Count
0,Total Primary Surgeries,321


---
## Analysis 2 · Graft Health Trends

In [31]:
# Build per-period graft summaries at multiple granularities
graft_summaries = {name: {} for name in
    ['overall', 'campus', 'surgery', 'surgeon', 'surgeon_surgery', 'campus_surgery']}

GROUP_SPECS = {
    'overall':         None,
    'campus':          ['sap_code'],
    'surgery':         ['surg_proc_group'],
    'surgeon':         ['surgeon_name'],
    'surgeon_surgery': ['surgeon_name', 'surg_proc_group'],
    'campus_surgery':  ['sap_code', 'surg_proc_group'],
}

for period, (start_day, end_day) in FOLLOW_UP_PERIODS.items():
    window_df = visit_df_primary[
        visit_df_primary['days_after_surgery'].between(start_day, end_day)
    ].copy()

    window_graft = (
        window_df.dropna(subset=['graft_health_text'])
        .sort_values(['id', 'visit_date'])
        .groupby('id', as_index=False).last()
    )

    window_graft = (
        surgery_df[['id', 'sap_code', 'surg_proc_group', 'surgeon_name']]
        .merge(window_graft[['id', 'graft_health_text']], on='id', how='left')
    )
    window_graft['graft_health_text'] = window_graft['graft_health_text'].fillna('NOT AVAILABLE')

    for name, cols in GROUP_SPECS.items():
        graft_summaries[name][period] = make_graft_summary(window_graft, cols)

# Combine across periods
EXCEL_SHEETS["2_Graft Overall"]      = combine_graft_periods(graft_summaries['overall'],        [], surgery_df)
EXCEL_SHEETS["2_Graft by Campus"]    = combine_graft_periods(graft_summaries['campus'],         ['sap_code'], surgery_df)
EXCEL_SHEETS["2_Graft by Surgery"]   = combine_graft_periods(graft_summaries['surgery'],        ['surg_proc_group'], surgery_df)
EXCEL_SHEETS["2_Graft by Surgeon"]   = combine_graft_periods(graft_summaries['surgeon'],        ['surgeon_name'], surgery_df)
EXCEL_SHEETS["2_Graft Srgn×Surg"]    = combine_graft_periods(graft_summaries['surgeon_surgery'],['surgeon_name', 'surg_proc_group'], surgery_df)
EXCEL_SHEETS["2_Graft Camp×Surg"]    = combine_graft_periods(graft_summaries['campus_surgery'], ['sap_code', 'surg_proc_group'], surgery_df)

print("✓ Analysis 2 (Graft Trends) — 6 sheets queued.")
display(EXCEL_SHEETS["2_Graft Overall"])

✓ Analysis 2 (Graft Trends) — 6 sheets queued.


,graft_health_text,Total Performed,1D Count,1D %,1W Count,1W %,1M Count,1M %,3M Count,3M %
0,ACUTE ALLOGRAFT REJECTION,321,0,0.00,0,0.00,3,0.93,1,0.31
1,CLEAR,321,127,39.56,118,36.76,163,50.78,47,14.64
2,EDEMA,321,121,37.69,54,16.82,37,11.53,11,3.43
3,FAILED,321,0,0.00,0,0.00,10,3.12,10,3.12
4,HAZY,321,35,10.90,34,10.59,24,7.48,4,1.25
5,INFILTRATE/INFECTION,321,0,0.00,0,0.00,1,0.31,0,0.00
6,LOOSE SUTURES,321,0,0.00,0,0.00,2,0.62,1,0.31
7,NOT ASSESSED,321,5,1.56,1,0.31,0,0.00,0,0.00
8,NOT AVAILABLE,321,33,10.28,114,35.51,77,23.99,244,76.01
9,SCAR,321,0,0.00,0,0.00,4,1.25,2,0.62


---
## Analysis 3 · Follow-Up Adherence

In [32]:
# ── Build per-surgery adherence table ──
today = pd.Timestamp.today().normalize()
summary_rows = []

for surgery_id, grp in visit_df_primary.groupby("id"):
    surg_date = grp["surg_date"].iloc[0]
    row = {
        "id": surgery_id,
        "gender": grp["gender"].iloc[0],
        "sub_category": grp["sub_category"].iloc[0],
        "surg_proc_group": grp["surg_proc_group"].iloc[0],
        "sap_code": grp["sap_code"].iloc[0],
        "age_category": grp["age_category"].iloc[0],
        "surg_age": grp["surg_age"].iloc[0],
    }
    visits = grp["days_after_surgery"].dropna()
    for period, (start, end) in FOLLOW_UP_PERIODS.items():
        visit_count = visits.between(start, end).sum()
        row[f"{period}_Visits"] = int(visit_count)
        window_end = surg_date + pd.Timedelta(days=end)
        if visit_count > 0:
            status = "YES"
        elif today <= window_end:
            status = "PENDING"
        else:
            status = "NO"
        row[period] = status
    summary_rows.append(row)

adherence_df = pd.DataFrame(summary_rows)

# ── Compute adherence summaries ──
adh_groups = {
    "Overall":      None,
    "Gender":        "gender",
    "Sub Category":  "sub_category",
    "Procedure":     "surg_proc_group",
    "SAP Code":      "sap_code",
    "Age Category":  "age_category",
}

for label, gcol in adh_groups.items():
    raw = adherence_summary(adherence_df, gcol)
    gcols = [gcol] if gcol else None
    combined = combine_adherence_periods(raw, adherence_df, group_cols=gcols)
    sheet_name = f"3_Adherence {label}"
    EXCEL_SHEETS[sheet_name] = combined

# ── Average visits by procedure ──
EXCEL_SHEETS["3_Avg Visits by Proc"] = average_visits(visit_df_primary, ["surg_proc_group"])

print("✓ Analysis 3 (Follow-Up Adherence) — 7 sheets queued.")
display(EXCEL_SHEETS["3_Adherence Overall"])

✓ Analysis 3 (Follow-Up Adherence) — 7 sheets queued.


,Status,Total Surgeries,1D Count,1D %,1W Count,1W %,1M Count,1M %,3M Count,3M %
0,NO,321,0,0.0,82,25.5,36,11.2,0,0.0
1,PENDING,321,0,0.0,0,0.0,0,0.0,219,68.2
2,YES,321,321,100.0,239,74.5,285,88.8,102,31.8


---
## Analysis 4 · Failed & Infection Cases

In [33]:
def extract_graft_cases(visit_df_primary: pd.DataFrame, status: str) -> pd.DataFrame:
    return (
        visit_df_primary[
            (visit_df_primary["days_after_surgery"] >= 0) &
            (visit_df_primary["graft_health_text"] == status)
        ]
        .drop_duplicates("tp_mrno_corrected")
        .sort_values("tp_mrno_corrected")
        [["tp_mrno_corrected", "tp_mrno", "id", "surg_proc_group", "sap_code",
          "surg_date", "visit_date", "days_after_surgery"]]
    )

failed_cases    = extract_graft_cases(visit_df_primary, "FAILED")
infection_cases = extract_graft_cases(visit_df_primary, "INFILTRATE/INFECTION")

EXCEL_SHEETS["4_Failed Cases"]    = failed_cases
EXCEL_SHEETS["4_Infection Cases"] = infection_cases

print(f"✓ Analysis 4 — Failed: {len(failed_cases)}, Infection: {len(infection_cases)} unique patients.")

✓ Analysis 4 — Failed: 21, Infection: 5 unique patients.


---
## Analysis 5 · Repeat Surgery Counts

In [34]:
# ── Repeat surgeries ──
repeat_df = visit_df_primary[
    (visit_df_primary["days_after_surgery"] > 0) &
    (visit_df_primary["advise_surg"] == "YES") &
    (visit_df_primary["fup_surg_done"] == "YES")
].copy()
repeat_df["Repeat_Category"] = repeat_df["fup_done_surg_proc"].apply(categorize_repeat_surgery)

# Overall counts
overall_repeat = (
    repeat_df["Repeat_Category"].value_counts()
    .rename_axis("Repeat Surgery").reset_index(name="Count")
)
overall_repeat.loc[len(overall_repeat)] = ["Overall", len(repeat_df)]

# By primary surgery type
repeat_summary = (
    repeat_df.groupby(["surg_proc_group", "Repeat_Category"]).size()
    .unstack(fill_value=0)
)
repeat_summary["Overall"] = repeat_summary.sum(axis=1)
repeat_summary = repeat_summary.reset_index()

primary_counts = adherence_df.groupby("surg_proc_group").size()
repeat_summary["Total Primary"] = repeat_summary["surg_proc_group"].map(primary_counts)
repeat_summary["Repeat %"] = (repeat_summary["Overall"] / repeat_summary["Total Primary"] * 100).round(2)

# KP detail breakdown
kp_details = (
    repeat_df[repeat_df["Repeat_Category"] == "KP"]
    .groupby(["surg_proc_group", "fup_done_surg_proc"]).size()
    .reset_index(name="Count")
    .sort_values(["surg_proc_group", "Count"], ascending=[True, False])
)

# KP patient list
kp_patients = (
    repeat_df[repeat_df["Repeat_Category"] == "KP"]
    [["tp_mrno_corrected", "id", "surg_proc_group", "fup_done_surg_proc",
      "days_after_surgery", "visit_date"]]
    .sort_values(["surg_proc_group", "fup_done_surg_proc", "tp_mrno_corrected"])
)

EXCEL_SHEETS["5_Repeat Overall"]   = overall_repeat
EXCEL_SHEETS["5_Repeat by Surg"]   = repeat_summary
EXCEL_SHEETS["5_KP Details"]       = kp_details
EXCEL_SHEETS["5_KP Patients"]      = kp_patients

print("✓ Analysis 5 (Repeat Surgeries) — 4 sheets queued.")
display(overall_repeat)

✓ Analysis 5 (Repeat Surgeries) — 4 sheets queued.


,Repeat Surgery,Count
0,Others,105
1,REBUBBLING,18
2,WOUND_RESUTURING,7
3,KP,3
4,Overall,133


---
## Analysis 6 · VA Change (non-THPK surgeries)

In [35]:
VA_df = visit_df_primary[visit_df_primary['surg_proc_group'] != 'THPK'].copy()

# ── Preop VA (latest before surgery, with same-day fallback) ──
def _latest_va(source_df, condition):
    return (
        source_df[condition & source_df["best_va_logmar"].notna()]
        .sort_values(["id", "visit_date"])
        .groupby("id").tail(1)
        [["id", "visit_date", "best_va_logmar"]]
        .rename(columns={"visit_date": "preop_visit_date", "best_va_logmar": "preop_logmar"})
    )

preop_before   = _latest_va(VA_df, VA_df["visit_date"] < VA_df["surg_date"])
preop_same_day = _latest_va(VA_df, VA_df["visit_date"] == VA_df["surg_date"])
preop = preop_before.combine_first(preop_same_day.set_index("id")).reset_index()

# ── Base table ──
base = VA_df.groupby("id").first().reset_index()
keep = ["id", "sap_code", "tp_mrno_corrected", "tp_mrno", "gender",
        "sub_category", "surg_date", "surg_eye", "surg_proc_group", "surgeon_name"]
VA_summary = base[keep].merge(preop, on="id", how="left")

# ── Follow-up VA per window ──
for period, (start, end) in FOLLOW_UP_PERIODS.items():
    p = period.lower()
    temp = (
        VA_df[
            VA_df["days_after_surgery"].between(start, end) &
            VA_df["best_va_logmar"].notna()
        ]
        .sort_values(["id", "visit_date"])
        .groupby("id").tail(1)
        [["id", "visit_date", "best_va_logmar"]]
        .rename(columns={"visit_date": f"{p}_visit_date", "best_va_logmar": f"{p}_logmar"})
    )
    VA_summary = VA_summary.merge(temp, on="id", how="left")
    VA_summary[f"{p}_diff"] = VA_summary["preop_logmar"] - VA_summary[f"{p}_logmar"]

# ── Line-change categories ──
for period in FOLLOW_UP_PERIODS:
    p = period.lower()
    VA_summary[f"{p}_line_cat"] = VA_summary[f"{p}_diff"].apply(line_category)

# ── Aggregate summaries ──
diff_cols = [f"{p.lower()}_diff" for p in FOLLOW_UP_PERIODS]

avg_change_proc = (
    VA_summary.groupby("surg_proc_group")[diff_cols].mean().round(3).reset_index()
)
avg_change_campus_surg = (
    VA_summary.groupby(["sap_code", "surg_proc_group"])[diff_cols].mean().round(3).reset_index()
)

# Line-change crosstabs per window
line_summaries = {}
for period in FOLLOW_UP_PERIODS:
    p = period.lower()
    line_summaries[period] = (
        VA_summary
        .groupby(["surg_proc_group", f"{p}_line_cat"]).size()
        .unstack(fill_value=0).reset_index()
    )

EXCEL_SHEETS["6_VA Patient Detail"]    = VA_summary
EXCEL_SHEETS["6_VA Avg by Proc"]       = avg_change_proc
EXCEL_SHEETS["6_VA Avg Camp×Surg"]     = avg_change_campus_surg
EXCEL_SHEETS["6_VA Lines 1D"]          = line_summaries["1D"]
EXCEL_SHEETS["6_VA Lines 1W"]          = line_summaries["1W"]
EXCEL_SHEETS["6_VA Lines 1M"]          = line_summaries["1M"]

print("✓ Analysis 6 (VA Change) — 6 sheets queued.")
display(avg_change_proc)

✓ Analysis 6 (VA Change) — 6 sheets queued.


,surg_proc_group,1d_diff,1w_diff,1m_diff,3m_diff
0,EK,-0.425,0.074,0.376,0.282
1,KPRO,0.585,-0.267,0.519,NaN
2,LK,-0.274,-0.066,0.220,0.437
3,PK,0.308,0.532,0.624,0.764


---
## 📥  Export All Results to Excel

In [36]:
try:
    with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
        for sheet_name, frame in EXCEL_SHEETS.items():
            safe_name = sheet_name[:31]
            frame.to_excel(writer, sheet_name=safe_name, index=False)
        beautify_workbook(writer.book)
    saved_file = OUTPUT_FILE
except PermissionError:
    import os
    base, ext = os.path.splitext(OUTPUT_FILE)
    saved_file = f"{base}_styled{ext}"
    with pd.ExcelWriter(saved_file, engine='openpyxl') as writer:
        for sheet_name, frame in EXCEL_SHEETS.items():
            safe_name = sheet_name[:31]
            frame.to_excel(writer, sheet_name=safe_name, index=False)
        beautify_workbook(writer.book)
    print(f"⚠️ Note: '{OUTPUT_FILE}' is open in Excel. Saved styled version to:\n   {saved_file}\n")

print(f"\n✓ All {len(EXCEL_SHEETS)} sheets saved with executive formatting to:")
print(f"   {saved_file}")
print("\nSheets written:")
for i, name in enumerate(EXCEL_SHEETS, 1):
    print(f"  {i:2d}. {name}")



✓ All 31 sheets saved with executive formatting to:
   C:\Users\MohammadFarhan\Desktop\Raksha\morbidity\Morbidity_Analysis_Results_Apr26.xlsx

Sheets written:
   1. 1_Total
   2. 1_By Gender
   3. 1_By Campus
   4. 1_By SurgType
   5. 1_By Category
   6. 1_By Surgeon
   7. 2_Graft Overall
   8. 2_Graft by Campus
   9. 2_Graft by Surgery
  10. 2_Graft by Surgeon
  11. 2_Graft Srgn×Surg
  12. 2_Graft Camp×Surg
  13. 3_Adherence Overall
  14. 3_Adherence Gender
  15. 3_Adherence Sub Category
  16. 3_Adherence Procedure
  17. 3_Adherence SAP Code
  18. 3_Adherence Age Category
  19. 3_Avg Visits by Proc
  20. 4_Failed Cases
  21. 4_Infection Cases
  22. 5_Repeat Overall
  23. 5_Repeat by Surg
  24. 5_KP Details
  25. 5_KP Patients
  26. 6_VA Patient Detail
  27. 6_VA Avg by Proc
  28. 6_VA Avg Camp×Surg
  29. 6_VA Lines 1D
  30. 6_VA Lines 1W
  31. 6_VA Lines 1M
